# Wan 2.1 Video Server for Kaggle (brand-new, working)

Hosts **Wan-AI/Wan2.1-T2V-1.3B-Diffusers** (text-to-video) on a free Kaggle GPU and
exposes it on a **public URL** with the exact HTTP API CineForge's `colab` backend
already speaks - zero changes on the laptop side.

| Endpoint | Method | What |
|---|---|---|
| `/health` | GET | status: model, mode, VRAM, busy flag |
| `/generate` | POST | text-to-video: JSON body → MP4 bytes |
| `/image` | POST | image-to-video → honest 501 (I2V is a 14B model, needs ~30 GB VRAM; use kling/seedance) |

**Why the previous notebook never hosted anything:** it pinned `diffusers==0.31.0`,
but `WanPipeline` / `AutoencoderKLWan` were only added in diffusers **0.33.0** - the
import always failed. This notebook pins **0.35.2** and builds the pipeline with a
VRAM strategy that actually fits Kaggle hardware (see Cell 3).

## ⚙️ STEP 0 - Kaggle settings (before running anything)

Settings panel (gear icon, top right):

| Setting | Value |
|---|---|
| **Accelerator** | **GPU T4 x2** (preferred - the code splits the model across both GPUs) |
| **Internet** | **ON** (required for pip + the one-time model download) |

## ▶️ How to run

1. Set the two settings above, **Save**.
2. **Run All** (top right ▶ menu). First run downloads ~29 GB of weights (5-20 min)
   and then stores a compact fp16 copy in `/kaggle/working`, so later sessions start
   in ~2 minutes.
3. Cell 6 prints a **PUBLIC URL** - paste it into `.env` as `COLAB_BASE_URL=<url>`.
4. Optionally leave Cell 7 (keep-alive) running so Kaggle doesn't reap an idle session.

Keep the Kaggle tab open. If the session dies, Run All again and update the URL in
`.env` (tunnel URLs change every run).


In [ ]:
# =============================================================================
#  CELL 1 - Install pinned dependencies.
#
#  torch and protobuf are NOT touched (we use Kaggle's CUDA-matched torch build
#  and Kaggle ships a working protobuf). Reinstalling either is how CUDA
#  mismatches and google-cloud breakages happen.
# =============================================================================
import subprocess
import sys

PKGS = [
    # Wan support (WanPipeline, AutoencoderKLWan) requires diffusers >= 0.33
    "diffusers==0.35.2",
    "transformers==4.55.4",
    "accelerate==1.10.1",
    "sentencepiece==0.2.0",
    "safetensors>=0.4.5",
    "imageio-ffmpeg>=0.5.1",
    "fastapi>=0.115.0",
    "uvicorn>=0.30.0",
    "pydantic>=2.7",
    "psutil>=5.9",
    "requests>=2.31",
    # only used automatically on small-RAM single-GPU sessions
    "bitsandbytes>=0.45.0",
]

print("Installing pinned stack ...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", *PKGS],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
print(r.stderr[-1500:])
print("pip exit code:", r.returncode)
assert r.returncode == 0, "pip install failed - is Settings -> Internet ON?"

# Verify protobuf works (Kaggle ships its own - we did NOT touch it).
import google.protobuf as _gp
if not hasattr(_gp, "runtime_version"):
    raise SystemExit(
        "google.protobuf has no runtime_version - stale Kaggle image. "
        "Session -> Restart session, then Run All.")
print("protobuf", _gp.__version__, "(Kaggle system, untouched)")
print()
print("CELL 1 done - stack installed. Continue (no restart).")


In [ ]:
# =============================================================================
#  CELL 2 - Environment sanity check. Everything green or it stops here with
#  a message that names the exact fix.
# =============================================================================
import importlib.metadata as md

import torch

fails = []


def ok(msg):
    print("  ✅", msg)


def bad(msg):
    print("  ❌", msg)
    fails.append(msg)


for pkg, want in [("diffusers", "0.35.2"), ("transformers", "4.55.4"),
                  ("accelerate", "1.10.1"), ("protobuf", None),
                  ("sentencepiece", None), ("fastapi", None),
                  ("bitsandbytes", None)]:
    try:
        got = md.version(pkg)
        if want is None or got == want:
            ok(f"{pkg} {got}")
        else:
            bad(f"{pkg} {got} installed but {want} required - re-run Cell 1")
    except md.PackageNotFoundError:
        bad(f"{pkg} NOT installed - re-run Cell 1")

print("torch", torch.__version__, "| CUDA build", torch.version.cuda)
if not torch.cuda.is_available():
    bad("CUDA not visible - Settings: Accelerator = GPU T4 x2, Internet = ON, Save, Run All")
else:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        ok(f"cuda:{i} = {p.name} ({p.total_memory / 1e9:.1f} GB)")
    _ = torch.zeros(1, device="cuda")  # warm the CUDA context
    torch.cuda.synchronize()

import psutil

ram = psutil.virtual_memory().total / 1e9
ok(f"system RAM {ram:.1f} GB")

# Real imports - the exact symbols Cell 3 uses.
from diffusers import AutoencoderKLWan, WanPipeline  # noqa: F401
from transformers import AutoTokenizer, UMT5EncoderModel  # noqa: F401

ok("diffusers Wan imports OK (WanPipeline, AutoencoderKLWan)")
ok("transformers umT5 imports OK (AutoTokenizer, UMT5EncoderModel)")

if fails:
    raise SystemExit("🛑 Fix the ❌ lines above, then re-run this cell.")
print("\n🎉 ALL CHECKS PASSED - safe to continue to Cell 3.")


In [ ]:
# =============================================================================
#  CELL 3 - Download + load Wan2.1 T2V 1.3B and place it on the GPU(s).
#
#  Components (fp16): umT5 text encoder ~11.4 GB | DiT transformer ~2.9 GB |
#  VAE ~0.5 GB (kept in fp32 - the Wan VAE produces artifacts in fp16).
#
#  Placement strategy (picked automatically):
#    * 2 GPUs (Kaggle T4 x2): text encoder -> cuda:1, DiT+VAE -> cuda:0.
#      No offloading, no quantization, fastest per-request path.
#    * 1 GPU + >= 20 GB RAM : accelerate model-cpu-offload (one model on the
#      GPU at a time; a few extra seconds per request).
#    * 1 GPU + small RAM    : 8-bit umT5 text encoder (bitsandbytes).
#
#  First run downloads ~29 GB (fp32 checkpoint). It then saves an fp16 copy
#  to /kaggle/working so later sessions skip the download entirely.
# =============================================================================
import gc
import os
import shutil
import time

import psutil
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from transformers import AutoTokenizer, BitsAndBytesConfig, UMT5EncoderModel

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
LOCAL_DIR = os.path.join(WORK, "wan_t2v_1.3b_fp16")  # fp16 cache (~14.5 GB)
USE_CACHE = True  # set False if you need the whole /kaggle/working quota


def free_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


N_GPU = torch.cuda.device_count()
VRAM0 = torch.cuda.get_device_properties(0).total_memory / 1e9
RAM = psutil.virtual_memory().total / 1e9
print(f"{N_GPU} GPU(s) | cuda:0 = {VRAM0:.1f} GB | RAM = {RAM:.1f} GB")

t0 = time.time()
have_local = USE_CACHE and os.path.isdir(os.path.join(LOCAL_DIR, "transformer"))
src = LOCAL_DIR if have_local else MODEL_ID
print("loading from:", src)

if N_GPU == 1 and RAM < 20:
    # ---- small-RAM single GPU: 8-bit umT5 on the GPU ------------------------
    print("low-RAM single-GPU session -> 8-bit umT5 text encoder")
    bnb_cfg = BitsAndBytesConfig(load_in_8bit=True,
                                 bnb_8bit_compute_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
    text_encoder = UMT5EncoderModel.from_pretrained(
        MODEL_ID, subfolder="text_encoder", quantization_config=bnb_cfg,
        torch_dtype=torch.float16, device_map="cuda:0", low_cpu_mem_usage=True)
    vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae",
                                           torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(
        MODEL_ID, tokenizer=tokenizer, text_encoder=text_encoder, vae=vae,
        torch_dtype=torch.float16)
    pipe.transformer.to("cuda:0")
    pipe.vae.to("cuda:0")
    MODE = "8bit-text-encoder"
else:
    pipe = WanPipeline.from_pretrained(src, torch_dtype=torch.float16)

    if src == MODEL_ID and USE_CACHE:
        # save BEFORE any offload hooks wrap the modules
        try:
            if shutil.disk_usage(WORK).free / 1e9 > 16:
                print("saving fp16 copy to", LOCAL_DIR, "(one-time, ~14.5 GB) ...")
                pipe.save_pretrained(LOCAL_DIR)
            else:
                print("not enough disk for the fp16 cache - skipping")
        except Exception as e:
            print("fp16 cache save skipped:", e)

    pipe.vae.to(torch.float32)  # Wan VAE in fp16 -> artifacts/NaN

    if N_GPU >= 2:
        # ---- two GPUs: umT5 on cuda:1, DiT + VAE on cuda:0 ------------------
        pipe.text_encoder.to("cuda:1")
        pipe.transformer.to("cuda:0")
        pipe.vae.to("cuda:0")

        # encode_prompt() feeds the text encoder tensors on the *execution*
        # device (cuda:0). This pre-hook transparently moves them onto
        # whatever GPU the encoder actually lives on, so the split is safe.
        def _inputs_to_own_device(module, args, kwargs):
            dev = next(module.parameters()).device
            args = tuple(t.to(dev) if isinstance(t, torch.Tensor) else t
                         for t in args)
            kwargs = {k: (v.to(dev) if isinstance(v, torch.Tensor) else v)
                      for k, v in kwargs.items()}
            return args, kwargs

        pipe.text_encoder.register_forward_pre_hook(_inputs_to_own_device,
                                                    with_kwargs=True)

        class _TwoGPUWanPipeline(WanPipeline):
            @property
            def _execution_device(self):
                return torch.device("cuda:0")

        pipe.__class__ = _TwoGPUWanPipeline  # DiT + VAE run on cuda:0
        MODE = "2-gpu-split"
    else:
        pipe.enable_model_cpu_offload()
        MODE = "cpu-offload"

try:
    pipe.enable_vae_tiling()
except AttributeError:
    pass
pipe.set_progress_bar_config(disable=True)
free_mem()

print(f"\n✅ Wan ready in {time.time() - t0:.0f}s | mode = {MODE}")
print("   model   :", MODEL_ID)
for i in range(N_GPU):
    print(f"   cuda:{i} allocated: {torch.cuda.memory_allocated(i) / 1e9:.1f} GB")


In [ ]:
# =============================================================================
#  CELL 4 - HTTP API. Same contract as the Colab server, so CineForge's
#  `colab` backend (src/forge/backends/colab.py) works unchanged:
#
#    GET  /health    -> status JSON (200 = ready)
#    POST /generate  -> text-to-video, JSON body -> MP4 bytes
#    POST /image     -> 501 (I2V is 14B / ~30 GB VRAM - not on free Kaggle)
# =============================================================================
import gc
import io as _io  # noqa: F401
import os
import random
import tempfile
import threading
import time
from typing import Optional

import torch
from diffusers.utils import export_to_video
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import JSONResponse, Response
from pydantic import BaseModel, Field

app = FastAPI(title="CineForge Kaggle Wan Server")
GEN_LOCK = threading.Lock()

DEFAULT_NEGATIVE = (
    "cartoon, anime, illustration, painting, drawing, CGI, 3d render, plastic "
    "skin, blurry, low quality, worst quality, jpeg artifacts, deformed, extra "
    "limbs, missing fingers, bad hands, bad teeth"
)


def _snap(v, lo=64, step=16):
    return max(lo, (int(v) // step) * step)


def _num_frames(duration, fps):
    # Wan requires (num_frames - 1) % 4 == 0; snap to the nearest legal
    # value within 9..121 (unsanitized values like 80 crash the pipeline).
    k = max(2, min(30, round((duration * fps - 1) / 4)))
    return 4 * k + 1


def _frames_to_mp4(frames, fps, seed, elapsed, w, h, model):
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as f:
        tmp = f.name
    try:
        export_to_video(frames, tmp, fps=fps)
        with open(tmp, "rb") as f:
            data = f.read()
    finally:
        try:
            os.unlink(tmp)
        except OSError:
            pass
    return Response(content=data, media_type="video/mp4", headers={
        "X-Seed": str(seed), "X-Fps": str(fps), "X-Model": model,
        "X-Elapsed-S": str(elapsed), "X-Width": str(w), "X-Height": str(h),
    })


class GenBody(BaseModel):
    prompt: str
    negative_prompt: str = ""
    width: int = Field(default=832, ge=64, le=1280)
    height: int = Field(default=480, ge=64, le=1280)
    fps: int = Field(default=16, ge=5, le=30)
    duration: float = Field(default=5.0, ge=1.0, le=10.0)
    num_inference_steps: int = Field(default=25, ge=4, le=50)
    guidance_scale: float = Field(default=5.0, ge=0.0, le=15.0)
    seed: Optional[int] = None


@app.get("/")
def root():
    return {"service": "cineforge-kaggle-wan",
            "endpoints": ["/health", "/generate", "/image"]}


@app.get("/health")
def health():
    return {
        "status": "ok", "ready": True, "t2v_model": MODEL_ID, "mode": MODE,
        "i2v_available": False, "busy": GEN_LOCK.locked(),
        "vram_gb": {i: round(torch.cuda.memory_allocated(i) / 1e9, 2)
                    for i in range(torch.cuda.device_count())},
    }


@app.post("/generate")
def generate(body: GenBody):
    if not GEN_LOCK.acquire(timeout=5):
        return JSONResponse(status_code=503,
                            content={"error": "server busy - retry shortly"})
    try:
        seed = body.seed if body.seed is not None else random.randint(0, 2**31 - 1)
        gen = torch.Generator("cuda:0").manual_seed(seed)
        n_frames = _num_frames(body.duration, body.fps)
        w, h = _snap(body.width), _snap(body.height)
        t0 = time.time()
        try:
            out = pipe(
                prompt=body.prompt,
                negative_prompt=body.negative_prompt or DEFAULT_NEGATIVE,
                width=w, height=h, num_frames=n_frames,
                num_inference_steps=body.num_inference_steps,
                guidance_scale=body.guidance_scale,
                generator=gen, max_sequence_length=512,
            ).frames[0]
        except torch.cuda.OutOfMemoryError as e:
            free_mem()
            return JSONResponse(status_code=507, content={
                "error": "CUDA OOM", "detail": str(e)[:300],
                "hint": "lower resolution (e.g. 768x432) or duration"})
        return _frames_to_mp4(out, body.fps, seed,
                              round(time.time() - t0, 1), w, h, MODEL_ID)
    finally:
        GEN_LOCK.release()


@app.post("/image")
async def image(file: UploadFile = File(...), prompt: str = Form("")):
    return JSONResponse(status_code=501, content={
        "error": "image-to-video not available on this GPU",
        "hint": "Wan I2V is a 14B model (~30 GB VRAM). "
                "Use the kling or seedance backend for image-to-video."})


print("✅ API ready: GET /health | POST /generate | POST /image (501)")


In [ ]:
# =============================================================================
#  CELL 5 - Smoke test: 9 frames at 512x320 (~1 min on T4 x2). Proves the full
#  path (prompt -> umT5 -> DiT -> VAE -> MP4) before we open the tunnel.
# =============================================================================
import gc

import torch

gc.collect()
torch.cuda.empty_cache()
try:
    r = generate(GenBody(
        prompt="a photorealistic city street at dusk, people walking, cinematic",
        width=512, height=320, duration=1.0, fps=9,
        num_inference_steps=6, guidance_scale=5.0, seed=1))
    code = getattr(r, "status_code", 200)
    if code != 200:
        print("⚠️ smoke test returned", code, "-", r.body[:300])
    else:
        print(f"✅ smoke test OK: {len(r.body):,} bytes of MP4, "
              f"{r.headers.get('X-Elapsed-S')}s, seed {r.headers.get('X-Seed')}")
except Exception as e:
    print("⚠️ smoke test failed:", type(e).__name__, str(e)[:300])
finally:
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# =============================================================================
#  CELL 6 - Start the API server and expose it on a PUBLIC URL.
#  Kaggle has no built-in proxy, so this tunnel IS the hosting:
#    1. cloudflared (preferred - stable URL, no interstitial page)
#    2. localtunnel via npx (fallback)
#  It then GETs /health THROUGH the public URL to prove the tunnel works.
# =============================================================================
import os
import re
import shutil
import socket
import subprocess
import threading
import time

import requests
import uvicorn

PORT = 8000

server = uvicorn.Server(uvicorn.Config(
    app, host="0.0.0.0", port=PORT,
    log_level="warning", access_log=False, lifespan="off"))
threading.Thread(target=server.run, daemon=True).start()

_deadline = time.time() + 30
while time.time() < _deadline:
    try:
        socket.create_connection(("127.0.0.1", PORT), timeout=1).close()
        break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError(f"uvicorn did not bind :{PORT} - re-run this cell")
print(f"API server listening on 127.0.0.1:{PORT}")

ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
PUBLIC_URL = None
METHOD = None
failures = []

# --- 1) cloudflared ----------------------------------------------------------
try:
    cf = shutil.which("cloudflared") or "/tmp/cloudflared"
    if not os.path.exists(cf):
        import urllib.request
        print("downloading cloudflared (~45 MB) ...")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/"
            "cloudflared-linux-amd64", cf)
        os.chmod(cf, 0o755)
    with open("/tmp/cf.log", "w") as logf:
        subprocess.Popen([cf, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                          "--no-autoupdate"], stdout=logf, stderr=logf)
    end = time.time() + 30
    while time.time() < end and PUBLIC_URL is None:
        time.sleep(1)
        text = ANSI.sub("", open("/tmp/cf.log", errors="ignore").read())
        hit = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
        if hit:
            PUBLIC_URL, METHOD = hit.group(0), "cloudflared"
    if PUBLIC_URL is None:
        failures.append(("cloudflared", "no URL in /tmp/cf.log after 30s"))
except Exception as e:
    failures.append(("cloudflared", repr(e)))

# --- 2) localtunnel (npx, no global install) ---------------------------------
if PUBLIC_URL is None:
    try:
        proc = subprocess.Popen(["npx", "--yes", "localtunnel", "--port", str(PORT)],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True)
        end = time.time() + 45
        line = ""
        while time.time() < end and PUBLIC_URL is None:
            ch = proc.stdout.read(1)
            if not ch:
                time.sleep(0.3)
                continue
            if ch == "\n":
                if "your url is:" in line.lower():
                    cand = line.split(":", 1)[1].strip()
                    if cand.startswith("http"):
                        PUBLIC_URL, METHOD = cand, "localtunnel"
                line = ""
            else:
                line += ch
        if PUBLIC_URL is None:
            proc.terminate()
            failures.append(("localtunnel", "no URL after 45s"))
    except Exception as e:
        failures.append(("localtunnel", repr(e)))

print()
print("=" * 72)
if PUBLIC_URL:
    try:
        r = requests.get(PUBLIC_URL + "/health", timeout=25)
        verdict = (f"/health through the public URL -> HTTP {r.status_code} ✓"
                   if r.status_code == 200 else
                   f"/health through the public URL -> HTTP {r.status_code} "
                   "(cloudflare edge may need ~30s - re-run this cell)")
    except Exception as e:
        verdict = f"public URL not answering yet ({type(e).__name__}) - re-run this cell"
    print(f"✅ HOSTED via {METHOD}")
    print()
    print("   PUBLIC API URL:")
    print("  ", PUBLIC_URL)
    print()
    print("  ", verdict)
    print()
    print("   On your laptop, add to .env:")
    print(f"      COLAB_BASE_URL={PUBLIC_URL}")
    print("   then use the colab backend in the studio, or:")
    print('      python -m src.main gen --backend colab "your prompt"')
else:
    print("🛑 ALL TUNNELS FAILED:")
    for name, err in failures:
        print(f"   - {name}: {err}")
    print("   Check Settings -> Internet = ON, then re-run this cell.")
print("=" * 72)


In [ ]:
# =============================================================================
#  CELL 7 (optional) - Keep-alive: ping /health every 60 s so Kaggle does not
#  kill the session as idle. Leave it running (the spinner is normal);
#  interrupt it to stop. GPU sessions cap at ~12 h, hence the 11 h loop.
# =============================================================================
import time

import requests

end = time.time() + 11 * 3600
pings = 0
while time.time() < end:
    try:
        requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5)
        pings += 1
    except Exception:
        pass
    time.sleep(60)
print(f"keep-alive finished after {pings} pings - re-run Cells 6-7 for a fresh URL")


---

## Test it

From inside this notebook (after Cell 6):

```python
import requests
r = requests.post(f"{PUBLIC_URL}/generate", json={
    "prompt": "photorealistic zombies walking down an abandoned city street at dusk, fog",
    "duration": 5.0, "width": 832, "height": 480, "fps": 16,
}, timeout=1800)
print(r.status_code)
if r.status_code == 200:
    open("/kaggle/working/t2v_test.mp4", "wb").write(r.content)
```

From your laptop: put `COLAB_BASE_URL=<url>` (no trailing slash) in `.env`, then use the
**colab** backend in the studio or `python -m src.main gen --backend colab "..."`.

## Notes

* `/image` returns **501** on purpose: image-to-video Wan is a 14B model needing
  ~30 GB VRAM - free Kaggle GPUs (T4 x2, 16 GB each) cannot serve it. Use the
  **kling**/**seedance** backends for image-to-video.
* A generation at 832x480, 5 s, 25 steps takes a few minutes on T4 x2. If a request
  returns 507 (CUDA OOM), lower `width`/`height` or `duration`.
* The fp16 weight cache lives in `/kaggle/working/wan_t2v_1.3b_fp16` (~14.5 GB of the
  20 GB quota). Set `USE_CACHE = False` in Cell 3 if you need that space for outputs.
